# Imports and setup for the notebook

In [38]:
# This loads all the libraries needed to execute the following cells
import ast
import concurrent
import html
import io
import os
import re
import shutil
import time
from urllib.parse import urlparse

import fasttext
import open_clip
import pandas as pd
import requests
# %pip install open-clip-torch torch torchvision pandas pillow
# %pip install webdavclient3
# %pip install accelerate qwen-vl-utils torchvision
%pip install tabulate
import torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from tqdm.notebook import tqdm

os.environ["HF_TOKEN"] = "YOUR_HUGGING_FACE_TOKEN"

BATCH_SIZE = 128
NUM_WORKERS = 4


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 1. Data Cleaning

In [9]:
# This is used to classify the images into 3 categories where we essentially only need to look at the chart_or_graph results

# --- Configuration ---
ROOT_DIR = "PLATFORM/PLATFORM_data"
CSV_PATH = "PLATFORM/PLATFORM_data/folder/posts.csv"

PLATFORM = ROOT_DIR.split('/')[0].capitalize()
OUTPUT_CSV = f"{PLATFORM}_classified_dataset.csv"
DEST_BASE_DIR = "classified_images"
BATCH_SIZE = 32
NUM_WORKERS = 4

# --- Setup Device & Model ---
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model = model.to(device)
tokenizer = open_clip.get_tokenizer('ViT-B-32')

candidate_labels = [
    "a data chart, graph, or plot",
    "a photograph of a real scene",
    "a document, text page, or interface screenshot"
]
category_names = ["chart_or_graph", "photograph", "document_or_screenshot"]
text_tokens = tokenizer(candidate_labels).to(device)

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

ImageFile.LOAD_TRUNCATED_IMAGES = True

def parse_image_path(raw_path_val, root_dir):
    """
    Parses single paths, stringified Python lists, or NaN values,
    and prepends root_dir if necessary to resolve real files.
    """
    if pd.isna(raw_path_val):
        return None

    path_str = str(raw_path_val).strip()
    if not path_str or path_str == "None" or path_str == "[]":
        return None

    if path_str.startswith("[") and path_str.endswith("]"):
        try:
            parsed_list = ast.literal_eval(path_str)
            if parsed_list and isinstance(parsed_list, list):
                path_str = str(parsed_list[0]).strip()
            else:
                return None
        except Exception:
            path_str = path_str.strip("[]'\" ").split(",")[0].strip()

    if not os.path.isabs(path_str) and not os.path.exists(path_str):
        joined_path = os.path.join(root_dir, path_str)
        if os.path.exists(joined_path):
            return joined_path

    return path_str if os.path.exists(path_str) else None


class DynamicDataset(Dataset):
    def __init__(self, df, transform=None, root_dir=""):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.root_dir = root_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        raw_val = row.get("local_image_paths")
        img_path = parse_image_path(raw_val, self.root_dir)

        if not img_path:
            dummy_tensor = torch.zeros((3, 224, 224))
            return dummy_tensor, False

        try:
            with Image.open(img_path) as img:
                img_rgb = img.convert("RGB")
                image_tensor = self.transform(img_rgb) if self.transform else img_rgb
            return image_tensor, True

        except Exception as e:
            print(e)
            dummy_tensor = torch.zeros((3, 224, 224))
            return dummy_tensor, False


# --- Dataset & DataLoader ---
df = pd.read_csv(CSV_PATH)
dataset = DynamicDataset(df, preprocess, root_dir=ROOT_DIR)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

# --- Inference ---
classified_types = []
model.eval()
with torch.no_grad():
    for images, valids in tqdm(dataloader, desc="Classifying Images"):
        images = images.to(device)

        image_features = model.encode_image(images)
        image_features /= image_features.norm(dim=-1, keepdim=True)

        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        probs = similarity.cpu().numpy()

        for valid, prob in zip(valids, probs):
            is_valid = valid.item() if isinstance(valid, torch.Tensor) else bool(valid)

            if not is_valid:
                classified_types.append("corrupt_or_missing")
            else:
                best_idx = prob.argmax()
                classified_types.append(category_names[best_idx])

df["classified_type"] = classified_types
df.to_csv(OUTPUT_CSV, index=False)
print("\nClassification complete!")
print("--- Final Classification Breakdown ---")
print(df["classified_type"].value_counts())

# --- Prepare Destination Directory ---
categories = ["chart_or_graph", "photograph", "document_or_screenshot"]
for cat in categories:
    os.makedirs(os.path.join(DEST_BASE_DIR, cat), exist_ok=True)


# --- Move/Copy Images ---
def sort_images_into_folders(csv_path, move_files=False):
    """
    Reads the output CSV and copies (or moves) images to category subfolders.
    Set move_files=True to MOVE instead of COPY.
    """
    df = pd.read_csv(csv_path)
    print(f"Sorting {len(df)} images into '{DEST_BASE_DIR}'...")

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Organizing Images"):
        raw_val = row.get("local_image_paths")
        src_path = parse_image_path(raw_val, ROOT_DIR)
        category = row.get("classified_type", "corrupt_or_missing")

        if pd.isna(category):
            category = "corrupt_or_missing"

        if src_path and os.path.exists(src_path):
            filename = os.path.basename(src_path)

            dest_path = os.path.join(DEST_BASE_DIR, category, filename)

            if move_files:
                shutil.move(src_path, dest_path)
            else:
                shutil.copy2(src_path, dest_path)


# Run sorting after classification completes
sort_images_into_folders(OUTPUT_CSV, move_files=False)
print("All images organized successfully!")

Using device: cpu


Classifying Images: 100%|██████████| 32/32 [01:16<00:00,  2.38s/it]



Classification complete!
--- Final Classification Breakdown ---
classified_type
corrupt_or_missing        695
photograph                154
document_or_screenshot    115
chart_or_graph             38
Name: count, dtype: int64
Sorting 1002 images into 'classified_images'...


Organizing Images: 100%|██████████| 1002/1002 [00:00<00:00, 3609.29it/s]

All images organized successfully!


# 2. Data Splitting and manual classification

In [10]:
# This is used to perform the split of the already filtered data into the chart_or_graph category. Additionally, this checks if the url and the image are actually valid.
# The result of this is the csv file, which we manually annotate and then inspect with the following cells.

# --- Configuration ---
LOCAL_IMAGE_DIR = "classified_images/chart_or_graph"
SAMPLE_OUT_DIR = "sampled_images"
INPUT_CLASSIFIED_CSV = f"{PLATFORM}_classified_dataset.csv"
ANNOTATION_CSV = "sampled_images_to_classify.csv"
PLATFORMS = ["bluesky", "telegram", "reddit"]
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

def check_url(platform, url):
    if not url or pd.isna(url) or not str(url).strip():
        return False, "empty url"
    try:
        if platform == "telegram":
            if not any(domain in url for domain in ["t.me/", "telegram.me/"]):
                return False, "not a Telegram URL"
            r = requests.get(url, headers=HEADERS, timeout=10, allow_redirects=True)
            if r.status_code != 200: return False, f"HTTP {r.status_code}"
            if not any(m in r.text.lower() for m in ["tgme_page", "tgme_widget_message", "telegram"]):
                return False, "not a Telegram post"
            return True, ""
        else:
            with requests.get(url, headers=HEADERS, timeout=10, stream=True) as r:
                if r.status_code != 200: return False, f"HTTP {r.status_code}"
                data = r.raw.read(20_000_001)
                if len(data) > 20_000_000: return False, "exceeds 20MB"
                Image.open(io.BytesIO(data)).verify()
                return True, ""
    except Exception as e:
        return False, f"failed: {e}"

def extract_first_path(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    # Handles string lists like "['path1', 'path2']" or single paths
    path_str = str(val).strip("[]'\" ").split(",")[0].strip(" '\"")
    return os.path.basename(path_str)

def extract_first_url(val):
    if pd.isna(val) or not str(val).strip():
        return ""
    return str(val).strip("[]'\" ").split(",")[0].strip(" '\"")

def sample_valid(platform_pool, platform, skipped_records):
    pool = platform_pool.sample(frac=1, random_state=42).reset_index(drop=True)
    selected = []

    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(check_url, platform, row["original_image_url"]): row for _, row in pool.iterrows()}
        for future in concurrent.futures.as_completed(futures):
            row = futures[future]
            ok, reason = future.result()
            if ok and len(selected) < min(len(pool), 3):
                selected.append(row)
            elif not ok:
                skipped_records.append({**row.to_dict(), "reason": reason})
            if len(selected) == min(len(pool), 3):
                break
    return pd.DataFrame(selected)

def main():
    if not os.path.exists(INPUT_CLASSIFIED_CSV):
        raise FileNotFoundError(f"Cannot find input CSV: {INPUT_CLASSIFIED_CSV}")

    df = pd.read_csv(INPUT_CLASSIFIED_CSV, low_memory=False)

    # Filter to target category if classified_type exists
    if "classified_type" in df.columns:
        df = df[df["classified_type"] == "chart_or_graph"].copy()

    # Extract single filename and URL from the dataset columns
    df["filename"] = df["local_image_paths"].apply(extract_first_path)
    df["original_image_url"] = df["image_urls"].apply(extract_first_url)

    # Drop rows without valid image filenames
    df = df[df["filename"] != ""].copy()

    # Detect platform or default to bluesky/telegram/reddit based on URL or URI
    def detect_platform(row):
        combined = f"{row.get('post_uri', '')} {row.get('original_image_url', '')}".lower()
        for p in PLATFORMS:
            if p in combined:
                return p
        return "bluesky" # Default fallback for at:// URIs

    df["platform"] = df.apply(detect_platform, axis=1)

    # Sample valid URLs per platform
    skipped_records, sampled_dfs = [], []
    for platform in PLATFORMS:
        pool = df[df["platform"] == platform]
        if not pool.empty:
            sampled_dfs.append(sample_valid(pool, platform, skipped_records))

    if not sampled_dfs:
        print("No valid sampled records found.")
        return

    df_sample = pd.concat(sampled_dfs, ignore_index=True)

    # Prepare annotation columns & copy image files
    os.makedirs(SAMPLE_OUT_DIR, exist_ok=True)
    for col in ["vis or not", "real or mis", "opinionated", "manual_topic", "notes", "isFromReputableSource"]:
        df_sample[col] = ""

    copied_count = 0
    for f in df_sample["filename"]:
        src = os.path.join(LOCAL_IMAGE_DIR, f)
        if os.path.exists(src):
            print(src)
            shutil.copy2(src, os.path.join(SAMPLE_OUT_DIR, f))
            copied_count += 1
        else:
            print(src)
            print(f"Warning: Local image file '{f}' not found in '{LOCAL_IMAGE_DIR}'")

    df_sample.to_csv(ANNOTATION_CSV, index=False)
    print(f"Done! Sampled {len(df_sample)} posts, copied {copied_count} images. Output: {ANNOTATION_CSV}")

if __name__ == "__main__":
    main()

classified_images/chart_or_graph/3ko7oumd4os2t_img_0.jpg
classified_images/chart_or_graph/3mgu65qkeqs2x_img_0.jpg
classified_images/chart_or_graph/3m5droujs522v_img_0.jpg
Done! Sampled 3 posts, copied 3 images. Output: sampled_images_to_classify.csv


# 3. Analysis of the manually inspected data

In [70]:
# This cell is used to perform the analysis of the manually inspected data

df = (pd.read_csv('results/images_classified_results.csv'))

# value_counts = df.groupby('platform')['vis or not'].value_counts()
# print(value_counts)


# Not all visualizations are proper for us to use, so after the first filter we also filter manually and mark them with T (data that was not used: tables only, numerical lists, images of Google Maps, thumbnails, images of too low resolution, etc.)
target_value = 'T'
filtered_df = df[df['vis or not'] == target_value].copy()


print(f"\nFiltered dataset size: {len(filtered_df)} rows")

filtered_df.to_csv('filtered_output.csv', index=False)
value_counts = filtered_df['manual_topic'].value_counts()
print(value_counts.head())
value_counts = filtered_df['platform'].value_counts()
print(value_counts)

value_counts = filtered_df.groupby('platform')['real or mis'].value_counts()
print(value_counts)
value_counts = filtered_df.groupby('platform')['opinionated'].value_counts()
print(value_counts)
value_counts = filtered_df.groupby('platform')['IsFromReputableSource'].value_counts()
print(value_counts)



cols = ['real or mis', 'opinionated', 'IsFromReputableSource']
combined_frames = []

for col in cols:
    counts = pd.crosstab(filtered_df['platform'], filtered_df[col])
    counts.columns = pd.MultiIndex.from_product([[col], counts.columns])
    combined_frames.append(counts)

total_df = pd.read_csv("results/all_platforms_image_manifest.csv")
total_counts = filtered_df['platform'].value_counts().to_frame(name='classified')
total_counts.columns = pd.MultiIndex.from_product([['classified'], ['Count']])
combined_frames.append(total_counts)

total_counts = total_df['platform'].str.lower().value_counts().to_frame(name='total_with_vis')
total_counts.columns = pd.MultiIndex.from_product([['total_with_vis'], ['Count']])
combined_frames.append(total_counts)

language_df = pd.read_csv("results/post_text_language_results_final.csv")

de_percentages = (
    language_df['language'].eq('de')
    .groupby(language_df['platform'].str.lower())
    .mean()
    .mul(100)
    .map(lambda x: f"{x:.1f}")
    .to_frame(name='de_pct')
)

de_percentages.columns = pd.MultiIndex.from_product([['german_posts'], ['%']])
combined_frames.append(de_percentages)

single_table = pd.concat(combined_frames, axis=1)
single_table.loc[['bluesky', 'reddit', 'telegram'], ('total_posts_images', 'Count')] = [9714, 25381, 66823]
single_table[('total_posts_images', 'Count')] = single_table[('total_posts_images', 'Count')]
display(single_table)

# styled_table = single_table.style.set_table_styles([{'selector': 'td, th', 'props': [('padding', '10px 25px')]}])
# display(styled_table)

pd.set_option('display.max_colwidth', None)
total_entries = filtered_df.groupby('platform').size().to_frame(name='Total Entries')
def get_top_topics(group):
    top5 = group['manual_topic'].value_counts().head(5)
    return "\n".join([f"• {topic}: {count}" for topic, count in top5.items()])

def get_top_channels(group):
    top5 = group['channel'].dropna().astype(str).str.strip().value_counts().head(5)
    return "\n".join([f"• {chan}" for chan in top5.index])

top_topics_series = filtered_df.groupby('platform', group_keys=False).apply(get_top_topics).to_frame(name='Top 5 Topics')
top_channels_series = filtered_df.groupby('platform'.lower(), group_keys=False).apply(get_top_channels).to_frame(name='Top 5 Channels/Subreddits')
new_summary_table = pd.concat([total_entries, top_topics_series, top_channels_series], axis=1).reset_index()
styled_table_classified = (
    new_summary_table.style
    .hide(axis='index')
    .set_properties(**{
        'text-align': 'left',
        'white-space': 'pre-wrap',
        'vertical-align': 'top'
    })
    .set_properties(subset=['platform'], **{
        'font-weight': 'bold',
        'vertical-align': 'middle'
    })
    .set_properties(subset=['Total Entries'], **{
        'text-align': 'center',
        'vertical-align': 'middle'
    })
)

display(styled_table_classified)


total_df['platform'] = total_df['platform'].str.lower()
total_entries = total_df.groupby('platform').size().to_frame(name='Total Entries')
top_channels_series = total_df.groupby('platform', group_keys=False).apply(get_top_channels).to_frame(name='Top 5 Channels/Subreddits')
new_summary_table = pd.concat([total_entries, top_channels_series], axis=1).reset_index()
styled_table_total = (
    new_summary_table.style
    .hide(axis='index')
    .set_properties(**{
        'text-align': 'left',
        'white-space': 'pre-wrap',
        'vertical-align': 'top'
    })
    .set_properties(subset=['platform'], **{
        'font-weight': 'bold',
        'vertical-align': 'middle'
    })
    .set_properties(subset=['Total Entries'], **{
        'text-align': 'center',
        'vertical-align': 'middle'
    })
)

display(styled_table_total)


# This creates the final csv, which has exactly the 200 classified samples per platform
filtered_df.to_csv("images_classified_results.csv", index=False)


Filtered dataset size: 600 rows
manual_topic
climate          219
living            59
Energy prices     51
politics          43
finance           30
Name: count, dtype: int64
platform
bluesky     200
reddit      200
telegram    200
Name: count, dtype: int64
platform  real or mis
bluesky   R              164
          M               36
reddit    R              141
          M               59
telegram  R              133
          M               67
Name: count, dtype: int64
platform  opinionated
bluesky   Y              102
          N               98
reddit    Y              104
          N               96
telegram  Y              127
          N               73
Name: count, dtype: int64
platform  IsFromReputableSource
bluesky   Y                        109
          N                         91
reddit    Y                        110
          N                         90
telegram  Y                        102
          N                         98
Name: count, dtype: int64


real or mis      opinionated      IsFromReputableSource       \
                   M    R           N    Y                     N    Y   
platform                                                                
bluesky           36  164          98  102                    91  109   
reddit            59  141          96  104                    90  110   
telegram          67  133          73  127                    98  102   

         classified total_with_vis german_posts total_posts_images  
              Count          Count            %              Count  
platform                                                            
bluesky         200           2340         97.0             9714.0  
reddit          200           1690         87.5            25381.0  
telegram        200           2744         75.0            66823.0

platform,Total Entries,Top 5 Topics,Top 5 Channels/Subreddits
bluesky,200,• climate: 95 • Energy prices: 38 • cars: 14 • climate/politics: 11 • politics: 10,• co2klima.bsky.social • klimakatastroph.bsky.social • nonviolence.bsky.social • hubsi2026.bsky.social • markanthon.bsky.social
reddit,200,• living: 53 • climate: 34 • finance: 26 • politics: 21 • mobility: 18,• r/austria • r/umwelt_de • r/finanzen • r/verkehrswende • r/wien
telegram,200,• climate: 90 • energy resources: 20 • politics: 12 • energy: 11 • immigration: 11,• EnergiewendeDeuschland • MissionDE • Migranten_Crime • betterknownow • tagesschauer


platform,Total Entries,Top 5 Channels/Subreddits
bluesky,2340,• co2klima.bsky.social • klimakatastroph.bsky.social • nonviolence.bsky.social • attac.at • klimavoracht.bsky.social
reddit,1690,• r/austria • r/umwelt_de • r/finanzen • r/wissenschaft • r/nachrichten
telegram,2744,• EnergiewendeDeuschland • MissionDE • Migranten_Crime • tagesschauer • betterknownow


In [37]:
# This is vibecoded for the language classification but the classification results have been checked manually again (at least sample wise)
# In the final version we have no NO_TEXT entries remaining as these have been manually classified

# ---------------------------------------------------------------- data
df = pd.read_csv("results/images_classified_results.csv")

# ---------------------------------------------------------------- language ID
model_path = "lid.176.bin"
if not os.path.exists(model_path):
    raise FileNotFoundError(
        "Download lid.176.bin first, e.g.: "
        "curl -L -O https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.bin"
    )
ft_model = fasttext.load_model(model_path)

MIN_CHARS = 10  # shorter texts (after cleaning) are labeled TOO_SHORT


def clean_for_langid(text):
    text = re.sub(r"https?://\S+", " ", text)   # URLs
    text = re.sub(r"[@#]\w+", " ", text)         # mentions / hashtags
    text = re.sub(r"\s+", " ", text)             # newlines, repeated spaces
    return text.strip()


def detect_language(text):
    """Return (lang_code, confidence). Works around the NumPy 2 bug in fasttext.predict."""
    try:
        labels, probs = ft_model.predict(text, k=1)
        return labels[0].replace("__label__", ""), float(probs[0])
    except ValueError:
        prob, label = ft_model.f.predict(text + "\n", 1, 0.0, "strict")[0]
        return label.replace("__label__", ""), float(prob)


# ---------------------------------------------------------------- HTTP helper
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}


def get(url, headers=HEADERS, params=None, retries=3):
    """GET with simple retry on HTTP 429 (rate limit)."""
    for attempt in range(retries):
        r = requests.get(url, headers=headers, params=params, timeout=20)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 5 * (attempt + 1)))
            time.sleep(min(wait, 60))
            continue
        r.raise_for_status()
        return r
    r.raise_for_status()
    return r


# ---------------------------------------------------------------- Bluesky
BSKY_API = "https://public.api.bsky.app/xrpc"
_did_cache = {}


def fetch_bluesky(post_url):
    m = re.search(r"/profile/([^/]+)/post/([^/?#]+)", post_url)
    if not m:
        raise ValueError("unrecognized Bluesky URL")
    actor, rkey = m.groups()

    if actor.startswith("did:"):
        did = actor
    else:
        if actor not in _did_cache:
            r = get(f"{BSKY_API}/com.atproto.identity.resolveHandle", params={"handle": actor})
            _did_cache[actor] = r.json()["did"]
        did = _did_cache[actor]

    uri = f"at://{did}/app.bsky.feed.post/{rkey}"
    r = get(f"{BSKY_API}/app.bsky.feed.getPosts", params={"uris": uri})
    posts = r.json().get("posts", [])
    if not posts:
        raise ValueError("POST_NOT_FOUND (deleted or not public)")
    return posts[0].get("record", {}).get("text", "")


# ---------------------------------------------------------------- Reddit
def _reddit_post_text(data):
    title = (data.get("title") or "").strip()
    body = (data.get("selftext") or "").strip()
    if body in ("[removed]", "[deleted]"):
        body = ""
    return (title + "\n" + body).strip()


# Reddit blocks direct access (403), so we read from the Arctic Shift archive,
# a public Reddit archive with a free API (no key). Data can lag live Reddit
# by hours, and very new or private posts may be missing.
ARCTIC = "https://arctic-shift.photon-reddit.com/api/posts/ids"
_reddit_cache = {}


def _reddit_id(url):
    m = re.search(r"/comments/([a-z0-9]+)", url)
    return m.group(1) if m else None


def prefetch_reddit(urls, batch=50):
    """Look up all Reddit posts in a few batched requests before the main loop."""
    ids = sorted({i for i in (_reddit_id(u) for u in urls) if i})
    for start in range(0, len(ids), batch):
        chunk = ids[start:start + batch]
        try:
            r = get(ARCTIC, params={"ids": ",".join(chunk)})
            payload = r.json()
            items = payload.get("data", []) if isinstance(payload, dict) else payload
            for p in items:
                _reddit_cache[p["id"]] = _reddit_post_text(p)
        except Exception as e:
            print(f"Arctic Shift batch {start // batch + 1} failed: {type(e).__name__}: {e}")
        time.sleep(1)
    print(f"Reddit: {len(_reddit_cache)} of {len(ids)} posts found in archive")


def fetch_reddit(post_url):
    post_id = _reddit_id(post_url)
    if not post_id:
        raise ValueError("unrecognized Reddit URL")
    if post_id in _reddit_cache:
        return _reddit_cache[post_id]
    raise ValueError("NOT_IN_ARCHIVE (Reddit blocks direct access; archive has no copy)")


# ---------------------------------------------------------------- Telegram
def _html_to_text(fragment):
    fragment = re.sub(r"<br\s*/?>", "\n", fragment)
    fragment = re.sub(r"<[^>]+>", "", fragment)
    return html.unescape(fragment).strip()


def fetch_telegram(post_url):
    base = post_url.split("?")[0]
    r = get(base + "?embed=1")
    m = re.search(
        r'<div class="tgme_widget_message_text[^"]*"[^>]*>(.*?)</div>',
        r.text, re.S,
    )
    if not m:
        return ""  # photo/video-only post, or not publicly embeddable
    return _html_to_text(m.group(1))


# ---------------------------------------------------------------- dispatcher
_cache = {}


def fetch_post_text(post_url):
    if post_url in _cache:
        return _cache[post_url]

    host = urlparse(post_url).netloc.lower()
    if "bsky.app" in host:
        text = fetch_bluesky(post_url); time.sleep(0.1)
    elif "reddit.com" in host:
        text = fetch_reddit(post_url)
    elif host in ("t.me", "telegram.me"):
        text = fetch_telegram(post_url); time.sleep(0.5)
    else:
        raise ValueError(f"UNSUPPORTED_PLATFORM ({host})")

    _cache[post_url] = text
    return text


# ---------------------------------------------------------------- per-row processing
def process_row(row):
    post_url = row["exact_post_url"]
    try:
        text = fetch_post_text(post_url)
    except Exception as e:
        return {"post_text": "", "language": "ERROR", "confidence": 0.0,
                "error": f"{type(e).__name__}: {e}"}

    if not text.strip():
        return {"post_text": "", "language": "NO_TEXT", "confidence": 0.0, "error": None}

    cleaned = clean_for_langid(text)
    if len(cleaned) < MIN_CHARS:
        return {"post_text": text, "language": "TOO_SHORT", "confidence": 0.0, "error": None}

    lang, conf = detect_language(cleaned)
    return {"post_text": text, "language": lang, "confidence": round(conf, 4), "error": None}


# ---------------------------------------------------------------- main loop
prefetch_reddit(df.loc[df["platform"] == "reddit", "exact_post_url"])

records = []
for i, row in df.iterrows():
    res = process_row(row)
    records.append(res)
    print(f"[{i + 1}/{len(df)}] {row.get('platform')} | {res['language']} "
          f"({res['confidence']}) {res['error'] or ''}")

out = pd.concat([df.reset_index(drop=True), pd.DataFrame(records)], axis=1)
os.makedirs("results", exist_ok=True)
out.to_csv("results/post_text_language_results.csv", index=False)
print(f"\nDone. {len(out)} rows saved to results/post_text_language_results.csv")

# ---------------------------------------------------------------- summary
NON_LANG = ["ERROR", "NO_TEXT", "TOO_SHORT"]
valid = out[~out["language"].isin(NON_LANG) & (out["confidence"] >= 0.5)]

print(f"\n{len(valid)} of {len(out)} posts had a reliably detected language\n")

print("Outcomes per platform (counts):")
print(out.groupby("platform")["language"]
        .apply(lambda s: s.where(s.isin(NON_LANG), "detected").value_counts())
        .unstack(fill_value=0))

print("\nLanguage share per platform (% of posts with detected language):")
print((pd.crosstab(valid["platform"], valid["language"], normalize="index") * 100).round(1))

print("\nOverall language share (%):")
print((valid["language"].value_counts(normalize=True) * 100).round(1))


Reddit: 191 of 191 posts found in archive
[1/600] bluesky | de (0.999) 
[2/600] bluesky | de (0.9761) 
[3/600] bluesky | de (0.9992) 
[4/600] bluesky | de (0.9974) 
[5/600] bluesky | de (0.9937) 
[6/600] bluesky | de (0.9955) 
[7/600] bluesky | de (0.9961) 
[8/600] bluesky | de (0.9943) 
[9/600] bluesky | de (0.9982) 
[10/600] bluesky | de (0.5032) 
[11/600] bluesky | de (0.9974) 
[12/600] bluesky | TOO_SHORT (0.0) 
[13/600] bluesky | de (0.3519) 
[14/600] bluesky | de (0.9939) 
[15/600] bluesky | de (0.9944) 
[16/600] bluesky | NO_TEXT (0.0) 
[17/600] bluesky | de (0.9904) 
[18/600] bluesky | en (0.9442) 
[19/600] bluesky | de (0.9976) 
[20/600] bluesky | de (0.9976) 
[21/600] bluesky | de (0.9961) 
[22/600] bluesky | de (0.9979) 
[23/600] bluesky | de (0.9938) 
[24/600] bluesky | de (0.9996) 
[25/600] bluesky | de (0.9922) 
[26/600] bluesky | de (0.9988) 
[27/600] bluesky | de (0.9998) 
[28/600] bluesky | de (0.994) 
[29/600] bluesky | de (0.9986) 
[30/600] bluesky | de (0.999) 
[31/